# 12-тарау. Нейрондық желіні нөлден жазамыз

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 12.1. Деректер: қолжазба цифрлар

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data / 16.0          # пиксельдерді [0, 1] аралығына келтіру
y = digits.target               # дұрыс жауаптар: 0–9

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print("Барлық деректер:", X.shape)
print("Оқыту:", X_train.shape, " Тест:", X_test.shape)

## 12.2. Желінің архитектурасы

## 12.3. Көмекші функциялар

In [ ]:
def relu(z):
    return np.maximum(0, z)

def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def one_hot(y, k=10):
    Y = np.zeros((len(y), k))
    Y[np.arange(len(y)), y] = 1
    return Y

print(one_hot(np.array([3, 0])))

## 12.4. NeuralNetwork класы

In [ ]:
class NeuralNetwork:
    def __init__(self, n_in, n_hidden, n_out, lr=0.1, seed=0):
        self.rng = np.random.default_rng(seed)
        # «He» баптауы: ReLU үшін ыңғайлы бастапқы салмақтар
        self.W1 = self.rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = self.rng.normal(0, np.sqrt(1 / n_hidden), (n_hidden, n_out))
        self.b2 = np.zeros(n_out)
        self.lr = lr

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1          # 1-қабат
        self.A1 = relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2    # 2-қабат
        self.P = softmax(self.Z2)                # ықтималдықтар
        return self.P

    def backward(self, Y):
        m = len(Y)
        dZ2 = (self.P - Y) / m                   # softmax + кросс-энтропия
        dW2 = self.A1.T @ dZ2
        db2 = dZ2.sum(axis=0)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * (self.Z1 > 0)       # ReLU туындысы: 1 немесе 0
        dW1 = self.X.T @ dZ1
        db1 = dZ1.sum(axis=0)
        # градиенттік түсу
        self.W2 -= self.lr * dW2;  self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1;  self.b1 -= self.lr * db1

    def loss(self, P, Y):
        return -np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))

    def predict(self, X):
        return self.forward(X).argmax(axis=1)

## 12.5. Оқыту циклі: шағын топтамалар

In [ ]:
def fit(self, X, y, epochs=30, batch_size=32, X_val=None, y_val=None):
    Y = one_hot(y)
    history = []
    for epoch in range(1, epochs + 1):
        order = self.rng.permutation(len(X))           # деректерді араластыру
        for start in range(0, len(X), batch_size):
            batch = order[start:start + batch_size]
            self.forward(X[batch])
            self.backward(Y[batch])
        # эпоха соңындағы көрсеткіштер
        loss = self.loss(self.forward(X), Y)
        train_acc = (self.predict(X) == y).mean()
        val_acc = (self.predict(X_val) == y_val).mean()
        history.append((epoch, loss, train_acc, val_acc))
        if epoch == 1 or epoch % 5 == 0:
            print(f"Эпоха {epoch:2d}: қате = {loss:.4f}, "
                  f"оқыту = {train_acc:.3f}, тест = {val_acc:.3f}")
    return history

NeuralNetwork.fit = fit     # әдісті класқа қосамыз

## 12.6. Желіні оқытамыз!

In [ ]:
net = NeuralNetwork(n_in=64, n_hidden=64, n_out=10, lr=0.1)
history = net.fit(X_train, y_train, epochs=30, batch_size=32,
                  X_val=X_test, y_val=y_test)

## 12.7. Нәтижелерді талдау

In [ ]:
from sklearn.metrics import confusion_matrix

pred = net.predict(X_test)
print("Тест дәлдігі:", (pred == y_test).mean().round(4))
cm = confusion_matrix(y_test, pred)
wrong = np.where(pred != y_test)[0]
print("Қате танылған суреттер саны:", len(wrong))

## 12.8. Желі нені үйренді?

In [ ]:
probs = net.forward(X_test[:1])[0]
print("Дұрыс жауап:", y_test[0])
for digit, p in enumerate(probs):
    bar = "█" * int(p * 40)
    print(f"{digit}: {p:6.1%} {bar}")

## 12.9. Нәтижені жақсарту